# Task 2: Chatbot for FAQs
A university student helpdesk chatbot built using NLP techniques.
User questions are matched against a FAQ dataset using TF-IDF vectorization 
and cosine similarity to return the most relevant answer.

In [1]:
# Installing required libraries for NLP processing and similarity matching
pip install nltk scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [25]:
# Building the FAQ dataset covering common university helpdesk topics
faq_data = [
    # Admissions
    ("What are the admission requirements?", "To apply, you need your matric and intermediate certificates, CNIC, domicile, and entry test result. Minimum 60% marks in intermediate are required."),
    ("When does the admission process start?", "Admissions typically open in July for the Fall semester and December for the Spring semester."),
    ("How do I apply for admission?", "You can apply online through the university's official website by filling out the admission form and uploading required documents."),
    ("How do I get admitted?", "You can apply online through the university's official website by filling out the admission form and uploading required documents."),
    ("How can I join the university?", "You can apply online through the university's official website by filling out the admission form and uploading required documents."),
    ("What do I need to apply?", "To apply, you need your matric and intermediate certificates, CNIC, domicile, and entry test result. Minimum 60% marks in intermediate are required."),
    ("Is there an entry test?", "Yes, the university conducts its own entry test. Some programs also accept HAT or SAT scores."),
    ("What is the last date to apply?", "The deadline varies by program. Please check the admissions portal for the latest schedule."),
    
    # Fees
    ("What is the tuition fee?", "Tuition fees vary by program. On average, fees range from PKR 30,000 to PKR 60,000 per semester."),
    ("Are there any scholarships available?", "Yes, the university offers merit-based and need-based scholarships. You can apply through the financial aid office."),
    ("How can I get a scholarship?", "Yes, the university offers merit-based and need-based scholarships. You can apply through the financial aid office."),
    ("What is the fee submission deadline?", "Fees must be submitted within two weeks of the semester start date to avoid a late fine."),
    ("When should I pay my fees?", "Fees must be submitted within two weeks of the semester start date to avoid a late fine."),
    ("Can I pay fees in installments?", "Yes, an installment plan is available. Contact the accounts office for details."),
    
    # Courses & Programs
    ("What programs does the university offer?", "The university offers programs in Engineering, Computer Science, Business, Social Sciences, and Medical Sciences at undergraduate and postgraduate levels."),
    ("How many credit hours are required to graduate?", "Most undergraduate programs require 130 to 136 credit hours to graduate."),
    ("How many courses do I need to graduate?", "Most undergraduate programs require 130 to 136 credit hours to graduate."),
    ("Can I change my major after admission?", "Yes, you can request a change of major within the first two semesters, subject to seat availability."),
    ("What is the duration of a bachelor's degree?", "A standard bachelor's degree takes 4 years to complete."),
    
    # Exams & Results
    ("When are the mid-term exams held?", "Mid-term exams are usually held in the 8th or 9th week of the semester."),
    ("How can I check my results?", "Results are published on the student portal. Log in with your student ID and password to view your grades."),
    ("Where can I check my grades?", "Results are published on the student portal. Log in with your student ID and password to view your grades."),
    ("What is the passing grade?", "The minimum passing grade is D, which corresponds to 40% marks in a course."),
    ("Can I apply for a re-checking of my paper?", "Yes, you can apply for re-checking within one week of result announcement by submitting a request to the examination office."),

    # Campus Life
    ("Is hostel accommodation available?", "Yes, separate hostels are available for male and female students. Apply through the student affairs office."),
    ("Does the university have a transport service?", "Yes, the university operates bus routes covering major areas of the city. Contact the transport office for routes and timings."),
    ("Are there student clubs and societies?", "Yes, the university has various clubs including coding, debate, sports, and arts societies. You can join during the societies fair at the start of each semester."),

    # General
    ("What are the university's office hours?", "The administrative offices are open Monday to Friday, 9 AM to 5 PM."),
    ("How do I contact my department?", "You can contact your department office directly or email them through the university's official website."),
    ("Where is the university located?", "The university is located in the heart of the city. Check the official website for the exact address and map."),
    ("Does the university offer online classes?", "Yes, some programs offer hybrid or fully online options. Check with your department for availability."),
]

print(f"Total FAQs loaded: {len(faq_data)}")

Total FAQs loaded: 31


In [26]:
# Imports for NLP preprocessing and similarity matching
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Preprocess text: lowercase, tokenize, remove stopwords and punctuation, lemmatize
def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and t not in string.punctuation]
    return " ".join(tokens)

# Extract questions and answers from dataset and preprocess all questions
questions = [pair[0] for pair in faq_data]
answers = [pair[1] for pair in faq_data]
processed_questions = [preprocess(q) for q in questions]

print("Text preprocessing complete.")
print(f"Sample processed question: {processed_questions[0]}")

Text preprocessing complete.
Sample processed question: admission requirement


[nltk_data] Downloading package punkt to C:\Users\Mohsin
[nltk_data]     Masood\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Mohsin
[nltk_data]     Masood\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Mohsin
[nltk_data]     Masood\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Mohsin
[nltk_data]     Masood\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [27]:
# Build TF-IDF matrix from preprocessed FAQ questions
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(processed_questions)

# Match user question to most similar FAQ using cosine similarity
def get_answer(user_question):
    processed_input = preprocess(user_question)
    input_vector = vectorizer.transform([processed_input])
    similarities = cosine_similarity(input_vector, tfidf_matrix)
    best_match_index = np.argmax(similarities)
    best_score = similarities[0][best_match_index]

    # Threshold: if no good match found, return fallback message
    if best_score < 0.2:
        return "I'm sorry, I couldn't find a relevant answer. Please contact the university directly."

    return answers[best_match_index]

# Quick test
print(get_answer("How do I apply?"))
print(get_answer("Tell me about scholarships"))

You can apply online through the university's official website by filling out the admission form and uploading required documents.
Yes, the university offers merit-based and need-based scholarships. You can apply through the financial aid office.


In [28]:
# Import Gradio for chat UI
import gradio as gr

# Handle chat interaction: maintain history and return updated conversation
def chat(user_input, history):
    if not user_input.strip():
        return history, ""
    answer = get_answer(user_input)
    history = history or []
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": answer})
    return history, ""

# Gradio chat UI layout
with gr.Blocks(title="University Helpdesk Chatbot") as app:
    gr.Markdown("# University Student Helpdesk Chatbot")
    gr.Markdown("Ask me anything about admissions, fees, courses, exams, or campus life.")

    # Chat history display
    chatbot = gr.Chatbot(label="Helpdesk Chat", height=400)

    # User input row with send button
    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Type your question here...",
            label="",
            scale=8,
            container=False
        )
        send_btn = gr.Button("Send", variant="primary", scale=1)

    # Clear conversation button
    clear_btn = gr.Button("Clear Chat", variant="secondary")

    # Button actions
    send_btn.click(fn=chat, inputs=[user_input, chatbot], outputs=[chatbot, user_input])
    user_input.submit(fn=chat, inputs=[user_input, chatbot], outputs=[chatbot, user_input])
    clear_btn.click(fn=lambda: ([], ""), outputs=[chatbot, user_input])

app.launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
